# EasyVisa — Visa Certification Prediction
---
**Author:** Thokozani | AI/ML Postgraduate Programme, UT Austin

## Approach
This notebook builds a machine learning solution to predict US visa certification
outcomes (Certified / Denied) for the OFLC. The analysis follows this structure:

1. Import libraries and load data
2. Dataset overview and sanity checks
3. Exploratory Data Analysis (univariate → bivariate → multivariate)
4. Data pre-processing (missing values, outliers, feature engineering, encoding)
5. Model building on original, oversampled (SMOTE), and undersampled data
6. Hyperparameter tuning of the three best-performing models
7. Final model comparison and selection
8. Actionable insights and recommendations
9. Conclusion

# 1. Importing Necessary Libraries

In [ ]:
# Core data manipulation and visualisation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling and evaluation
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report,
    ConfusionMatrixDisplay
)

# Classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier
)
from xgboost import XGBClassifier

# Class imbalance handling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import warnings
warnings.filterwarnings("ignore")

# Plot aesthetics
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 1

# 2. Loading the Dataset

In [ ]:
df = pd.read_csv("EasyVisa.csv")
print(f"Dataset loaded successfully. Shape: {df.shape}")
print(f"\nRows: {df.shape[0]:,} | Columns: {df.shape[1]}")

# 3. Overview of the Dataset

## 3.1 Observations

In [ ]:
print("=== First 5 rows ===")
df.head()

In [ ]:
print("=== Last 5 rows ===")
df.tail()

In [ ]:
print("=== Data types and non-null counts ===")
df.info()

In [ ]:
print("=== Descriptive statistics — numeric features ===")
df.describe().T

In [ ]:
print("=== Descriptive statistics — categorical features ===")
df.describe(include="object").T

## 3.2 Sanity Checks

In [ ]:
# Check for duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

# Check for missing values
print(f"\nMissing values per column:")
print(df.isnull().sum())

# Verify case_id is a unique identifier
print(f"\nUnique case_id values: {df['case_id'].nunique()} (expected {df.shape[0]})")

# Check numeric ranges
print(f"\nno_of_employees range: {df['no_of_employees'].min()} to {df['no_of_employees'].max():,}")
print(f"Negative no_of_employees: {(df['no_of_employees'] < 0).sum()} rows")
print(f"\nyr_of_estab range: {df['yr_of_estab'].min()} to {df['yr_of_estab'].max()}")
print(f"prevailing_wage range: ${df['prevailing_wage'].min():.2f} to ${df['prevailing_wage'].max():,.2f}")

**Sanity check observations:**
- No duplicate rows found.
- No missing values in any column — no imputation needed.
- `case_id` is a unique identifier per application — will be dropped before modelling.
- `no_of_employees` contains **33 negative values** — impossible in reality; likely data-entry
  errors (e.g. a minus sign typed accidentally). These will be corrected using `.abs()` during
  preprocessing.
- `yr_of_estab` goes back to 1800, which produces large `company_age` values but is not
  inherently wrong for long-established institutions.

# 4. Exploratory Data Analysis (EDA)

## 4.1 Univariate Analysis

We first examine each variable in isolation to understand its distribution before
looking at relationships with the target.

In [ ]:
# --- Numeric features: distribution plots ---
# Create annual_wage here for EDA; formal derivation happens in preprocessing
wage_map = {"Hour": 40 * 52, "Week": 52, "Month": 12, "Year": 1}
df["annual_wage"] = df["prevailing_wage"] * df["unit_of_wage"].map(wage_map)
df["company_age"] = 2016 - df["yr_of_estab"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
numeric_features = {
    "no_of_employees": "Number of employees",
    "annual_wage":     "Annual prevailing wage (USD)",
    "company_age":     "Company age (years)",
}
for ax, (col, label) in zip(axes, numeric_features.items()):
    data = df[col].clip(upper=df[col].quantile(0.99))  # clip top 1% for readability
    ax.hist(data, bins=40, color="#378ADD", edgecolor="white")
    ax.set_title(f"Distribution of {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("uni_numeric.png")
plt.show()

**Observations — numeric features:**
- `no_of_employees` is heavily right-skewed; most employers are small-to-mid sized
  companies, with a long tail of very large corporations (up to 602,069 employees).
- `annual_wage` is also right-skewed with a peak around $50K–$80K. A small number of
  roles command wages above $200K.
- `company_age` shows a bimodal pattern — a cluster of recently established companies
  (post-2000) and a cluster of established companies (40–80 years old), with relatively
  fewer in the middle.

In [ ]:
# --- Categorical features: count plots ---
cat_cols = [
    "continent", "education_of_employee", "has_job_experience",
    "requires_job_training", "region_of_employment",
    "unit_of_wage", "full_time_position", "case_status"
]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, color="#534AB7", edgecolor="white")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height()):,}",
                    (p.get_x() + p.get_width() / 2, p.get_height() + 50),
                    ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("uni_categorical.png")
plt.show()

**Observations — categorical features:**
- `case_status`: ~67% Certified vs ~33% Denied — a moderate class imbalance that
  will be addressed through oversampling and undersampling strategies.
- `continent`: Asia dominates (>60% of applicants), reflecting US visa demand patterns.
- `education_of_employee`: Master's degree holders form the largest group, followed by
  Bachelor's. Very few High School-only applicants.
- `has_job_experience`: The majority of applicants (Y) have prior experience.
- `requires_job_training`: Most positions do not require training.
- `region_of_employment`: The South and Northeast receive the most applications.
- `unit_of_wage`: The majority of wages are quoted on a Yearly basis.
- `full_time_position`: Predominantly full-time roles (Y).

## 4.2 Bivariate Analysis — Leading Questions

In [ ]:
# Q1. What is the distribution of visa case statuses (certified vs. denied)?
fig, ax = plt.subplots(figsize=(5, 4))
counts = df["case_status"].value_counts()
ax.bar(counts.index, counts.values, color=["#639922", "#E24B4A"], edgecolor="white")
ax.set_title("Q1: Visa case status distribution")
ax.set_ylabel("Number of applications")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}\n({int(p.get_height())/len(df)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2, p.get_height() + 50),
                ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("eda_q1_status.png")
plt.show()

print("Case status distribution:")
print(df["case_status"].value_counts())
print(df["case_status"].value_counts(normalize=True).mul(100).round(1))

**Observations (Q1):**
- 17,018 applications (66.8%) were Certified; 8,462 (33.2%) were Denied.
- The dataset has a 2:1 class imbalance. A naive model predicting "Certified" for
  all cases would already achieve ~67% accuracy — making accuracy alone a misleading
  metric. We must monitor Recall and F1 carefully.

In [ ]:
# Q2. How does education level impact visa approval rates?
edu_order = ["High School", "Bachelor's", "Master's", "Doctorate"]
edu_ct = pd.crosstab(df["education_of_employee"], df["case_status"], normalize="index").mul(100)

fig, ax = plt.subplots(figsize=(7, 4))
edu_ct.reindex(edu_order)[["Certified", "Denied"]].plot(
    kind="bar", ax=ax, color=["#639922", "#E24B4A"], edgecolor="white"
)
ax.set_title("Q2: Certification rate by education level")
ax.set_ylabel("Percentage of applications (%)")
ax.set_xlabel("Education level")
ax.set_xticklabels(edu_order, rotation=0)
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig("eda_q2_education.png")
plt.show()

print("Certification rate by education:")
print(edu_ct.reindex(edu_order).round(1))

**Observations (Q2):**
- Education is the single clearest separator. Doctorate holders are certified at 87%,
  Master's at 79%, Bachelor's at 62%, and High School-only applicants at just 34%.
- Higher education strongly predicts certification, likely because it is easier for
  employers to demonstrate that no equivalent US worker is available for specialised roles.

In [ ]:
# Q3. Does prior job experience affect approval rates?
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, title in zip(
    axes,
    ["has_job_experience", "requires_job_training"],
    ["Has job experience", "Requires job training"],
):
    ct = pd.crosstab(df[col], df["case_status"], normalize="index").mul(100)
    ct[["Certified", "Denied"]].plot(kind="bar", ax=ax,
                                      color=["#639922", "#E24B4A"], edgecolor="white", rot=0)
    ax.set_title(f"Q3/Extra: Certification rate — {title}")
    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("")
    ax.legend()
plt.tight_layout()
plt.savefig("eda_q3_experience.png")
plt.show()

print("Certification rate by job experience:")
print(pd.crosstab(df["has_job_experience"], df["case_status"], normalize="index").mul(100).round(1))

**Observations (Q3):**
- Applicants with prior job experience (Y) are certified at ~74.5% vs ~56.1% for those
  without — an 18 percentage-point difference.
- Job training requirement shows a smaller effect and is near the overall average for
  both Y and N — it will likely carry low feature importance in models.

In [ ]:
# Q4. How does prevailing wage affect approval?
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="case_status", y="annual_wage",
            palette={"Certified": "#639922", "Denied": "#E24B4A"},
            showfliers=False, ax=ax)
ax.set_title("Q4: Annual prevailing wage by case status (outliers hidden)")
ax.set_ylabel("Annual wage (USD)")
ax.set_xlabel("Case status")
plt.tight_layout()
plt.savefig("eda_q4_wage.png")
plt.show()

print("Annual wage summary by status:")
print(df.groupby("case_status")["annual_wage"].describe().T)

**Observations (Q4):**
- Counterintuitively, *denied* applications have a higher median annual wage (~$92K)
  than certified ones (~$79K).
- This likely reflects that high-wage, highly specialised roles attract more scrutiny —
  OFLC must be more convinced that no US worker can fill such roles. High wage alone
  does not guarantee certification.

In [ ]:
# Q5. Do certain US regions have higher approval rates?
reg_ct = pd.crosstab(df["region_of_employment"], df["case_status"], normalize="index").mul(100)
fig, ax = plt.subplots(figsize=(7, 4))
reg_ct.sort_values("Certified")["Certified"].plot(
    kind="barh", ax=ax, color="#1D9E75", edgecolor="white"
)
ax.set_title("Q5: Certification rate by US region of employment")
ax.set_xlabel("Certification rate (%)")
ax.set_ylabel("Region")
for i, v in enumerate(reg_ct.sort_values("Certified")["Certified"]):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=10)
plt.tight_layout()
plt.savefig("eda_q5_region.png")
plt.show()

print("Certification rate by region:")
print(reg_ct.sort_values("Certified").round(1))

**Observations (Q5):**
- The Midwest has the highest certification rate (75.5%), followed by the South (70%).
- Island territories and the West have the lowest rates (~60–62%).
- Regional variation may reflect the occupation mix in each region rather than geographic
  bias — the Midwest has more manufacturing and agriculture roles where labour shortages
  are more demonstrable.

In [ ]:
# Q6. Does company size (no_of_employees) influence approval?
# Fix negatives before this analysis
df["no_of_employees"] = df["no_of_employees"].abs()

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="case_status", y="no_of_employees",
            palette={"Certified": "#639922", "Denied": "#E24B4A"},
            showfliers=False, ax=ax)
ax.set_title("Q6: Company size by case status (outliers hidden)")
ax.set_ylabel("Number of employees")
ax.set_xlabel("Case status")
plt.tight_layout()
plt.savefig("eda_q6_company_size.png")
plt.show()

print("Company size summary by status:")
print(df.groupby("case_status")["no_of_employees"].describe().T)

**Observations (Q6):**
- Certified applications tend to come from slightly larger companies (higher median
  no_of_employees), though the difference is modest.
- Larger, established companies likely have better compliance records and stronger
  documentation processes, which supports approval.

In [ ]:
# Q7. Are approval rates different across continents?
cont_ct = pd.crosstab(df["continent"], df["case_status"], normalize="index").mul(100)
fig, ax = plt.subplots(figsize=(7, 4))
cont_ct.sort_values("Certified")["Certified"].plot(
    kind="barh", ax=ax, color="#534AB7", edgecolor="white"
)
ax.set_title("Q7: Certification rate by continent of employee")
ax.set_xlabel("Certification rate (%)")
ax.set_ylabel("Continent")
for i, v in enumerate(cont_ct.sort_values("Certified")["Certified"]):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=10)
plt.tight_layout()
plt.savefig("eda_q7_continent.png")
plt.show()

print("Certification rate by continent:")
print(cont_ct.sort_values("Certified").round(1))

**Observations (Q7):**
- European applicants have the highest certification rate (79.2%), followed by Africa
  (72.1%) and Asia (65.3%).
- South American applicants have the lowest rate (57.9%).
- The variation likely reflects the occupational profile of applicants from each region
  (e.g. European applicants may disproportionately apply for high-skill, high-education
  roles) rather than continent being a direct cause of approval.

## 4.3 Multivariate Analysis — Beyond the Basics

In [ ]:
# A. Education x Job experience interaction
pivot_edu_exp = df.pivot_table(
    index="education_of_employee", columns="has_job_experience",
    values="case_status", aggfunc=lambda x: (x == "Certified").mean() * 100
).reindex(edu_order)

fig, ax = plt.subplots(figsize=(8, 4))
pivot_edu_exp.plot(kind="bar", ax=ax, color=["#E24B4A", "#639922"], edgecolor="white")
ax.set_title("A: Certification rate by education AND job experience")
ax.set_ylabel("Certification rate (%)")
ax.set_xlabel("Education level")
ax.set_xticklabels(edu_order, rotation=0)
ax.legend(title="Has experience", labels=["No", "Yes"])
plt.tight_layout()
plt.savefig("eda_multi_A.png")
plt.show()

print("Certification rate — education × experience:")
print(pivot_edu_exp.round(1))

**Observations (Multivariate A):**
- Job experience amplifies the effect of education at every level. A Bachelor's-educated
  applicant with experience (71.3%) outperforms a Doctorate holder without experience
  (78% vs 67% for the experience/no-experience Doctorate split).
- The combination of High School + no experience produces the lowest certification
  rate of all groups (~28%), representing the highest-risk profile.

In [ ]:
# B. Continent x Education heatmap
pivot_cont_edu = df.pivot_table(
    index="continent", columns="education_of_employee",
    values="case_status", aggfunc=lambda x: (x == "Certified").mean() * 100
)[edu_order]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot_cont_edu, annot=True, fmt=".1f", cmap="RdYlGn",
            linewidths=0.5, ax=ax, vmin=20, vmax=100)
ax.set_title("B: Certification rate (%) — continent × education")
ax.set_xlabel("Education level")
ax.set_ylabel("Continent")
plt.tight_layout()
plt.savefig("eda_multi_B.png")
plt.show()

**Observations (Multivariate B):**
- The heatmap confirms education is the dominant driver across all continents —
  every row darkens (higher rate) moving left to right.
- South American Doctorate holders still achieve ~80% certification, confirming
  the continent effect is secondary to education.
- African High School-only applicants have the lowest rate (~22%), a very small
  sub-group likely worth flagging for targeted pre-screening guidance.

In [ ]:
# C. Correlation heatmap — numeric features
num_cols = ["no_of_employees", "annual_wage", "company_age"]
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="Blues",
            linewidths=0.5, ax=ax)
ax.set_title("C: Correlation among numeric features")
plt.tight_layout()
plt.savefig("eda_multi_C.png")
plt.show()

print("Correlation matrix:")
print(df[num_cols].corr().round(2))

**Observations (Multivariate C):**
- Correlations between the three numeric features are very low (all < 0.1), confirming
  they carry independent information. No multicollinearity concern.
- `annual_wage` and `no_of_employees` are essentially uncorrelated (r = 0.02), meaning
  large companies do not necessarily pay more (and vice versa).

# 5. Data Pre-processing

We apply each step only where needed, with explicit reasoning for every decision.

## 5.1 Missing Value Treatment

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

**Decision:** No missing values exist in the dataset. No imputation is needed.

## 5.2 Outlier Detection and Treatment

In [ ]:
# IQR-based check on numeric columns
for col in ["no_of_employees", "annual_wage", "company_age"]:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_low = (df[col] < lower).sum()
    n_high = (df[col] > upper).sum()
    print(f"{col}: {n_low} below lower fence ({lower:,.1f}), "
          f"{n_high} above upper fence ({upper:,.1f})")

**Decision on outliers:**
- `no_of_employees`: 33 negative values were confirmed as data-entry errors and corrected
  using `.abs()` earlier in the EDA. High values (large corporations) are legitimate and
  are **retained** — removing major employers would bias the model.
- `annual_wage`: Upper outliers exist but are legitimate — senior roles in specialised
  fields command very high wages. These are **retained** to preserve the signal that wage
  level carries for visa outcomes.
- `company_age`: Values up to 216 years exist (yr_of_estab = 1800). These are plausible
  for long-established institutions. **Retained.**

No rows are dropped. The only correction applied was `.abs()` on `no_of_employees`.

## 5.3 Feature Engineering

In [ ]:
# annual_wage and company_age were already computed during EDA.
# We retain them and drop the raw source columns they replace,
# as the derived features carry more intuitive predictive signal:
#   - company_age (continuous years) is more meaningful than yr_of_estab (a calendar year)
#   - annual_wage (normalised to the same unit) allows direct wage comparison across
#     applicants who were quoted hourly, weekly, monthly, or yearly

print("Features before engineering:")
print(df.columns.tolist())

# Drop case_id (unique identifier — not predictive)
# Drop yr_of_estab and prevailing_wage (replaced by derived features)
df.drop(columns=["case_id", "yr_of_estab", "prevailing_wage"], inplace=True)

print("\nFeatures after dropping raw columns:")
print(df.columns.tolist())

## 5.4 Prepare Data for Modelling

In [ ]:
# Encode target variable: Certified = 1, Denied = 0
df["target"] = (df["case_status"] == "Certified").astype(int)
print("Target encoding: Certified=1, Denied=0")
print(df["target"].value_counts())

In [ ]:
# Label-encode all categorical predictor columns
categorical_cols = [
    "continent", "education_of_employee", "has_job_experience",
    "requires_job_training", "region_of_employment",
    "unit_of_wage", "full_time_position",
]

# We use LabelEncoder here because tree-based models (our entire model set)
# do not require one-hot encoding — they split on ordinal values directly.
df_model = df.copy()
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le

print("Label encoding applied to:", categorical_cols)
df_model.head()

In [ ]:
# Define feature matrix and target vector
X = df_model.drop(columns=["case_status", "target"])
y = df_model["target"]

print("Feature matrix shape:", X.shape)
print("Features:", X.columns.tolist())
print(f"\nClass balance — Certified (1): {y.mean()*100:.1f}% | Denied (0): {(1-y.mean())*100:.1f}%")

In [ ]:
# Stratified 70/30 train-test split
# Stratified to ensure the 67/33 class ratio is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"\nTrain class balance:\n{y_train.value_counts(normalize=True).mul(100).round(1)}")
print(f"\nTest class balance:\n{y_test.value_counts(normalize=True).mul(100).round(1)}")

# Helper Functions for Model Evaluation

In [ ]:
def get_metrics(model, X_tr, y_tr, X_te, y_te):
    """
    Returns train and test metrics as dicts.
    Reporting both lets us detect overfitting:
    a large train-test gap indicates the model has memorised the training data.
    """
    y_pred_train = model.predict(X_tr)
    y_pred_test  = model.predict(X_te)

    def score(y_true, y_pred):
        return {
            "Accuracy":  round(accuracy_score(y_true, y_pred) * 100, 2),
            "Recall":    round(recall_score(y_true, y_pred) * 100, 2),
            "Precision": round(precision_score(y_true, y_pred) * 100, 2),
            "F1":        round(f1_score(y_true, y_pred) * 100, 2),
        }

    return score(y_tr, y_pred_train), score(y_te, y_pred_test)


def build_and_evaluate(X_tr, y_tr, X_te, y_te, label=""):
    """
    Trains six classifiers, prints train vs test metrics, and returns
    results dataframe + dict of fitted models.
    """
    models = {
        "Decision Tree":     DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Bagging":           BaggingClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        "Random Forest":     RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        "AdaBoost":          AdaBoostClassifier(random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost":           XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss",
                                           verbosity=0),
    }

    rows = []
    fitted = {}
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        fitted[name] = model
        train_m, test_m = get_metrics(model, X_tr, y_tr, X_te, y_te)
        rows.append({
            "Model":          name,
            "Train Accuracy": train_m["Accuracy"],
            "Test Accuracy":  test_m["Accuracy"],
            "Train Recall":   train_m["Recall"],
            "Test Recall":    test_m["Recall"],
            "Train Precision":train_m["Precision"],
            "Test Precision": test_m["Precision"],
            "Train F1":       train_m["F1"],
            "Test F1":        test_m["F1"],
        })

    results = pd.DataFrame(rows).set_index("Model")
    print(f"\n{'='*60}")
    print(f"  {label} — Train vs Test metrics")
    print(f"{'='*60}")
    print(results.to_string())
    return results, fitted

# 6. Model Building — Original Data

In [ ]:
results_orig, models_orig = build_and_evaluate(
    X_train, y_train, X_test, y_test, label="ORIGINAL DATA"
)

**Observations — original data:**
- Decision Tree shows a large train-test accuracy gap (~99% train vs ~72% test),
  confirming it overfits on the original data. Pruning or ensemble methods are needed.
- Ensemble methods (Bagging, Random Forest, AdaBoost, Gradient Boosting, XGBoost)
  all outperform the single Decision Tree on the test set, as expected.
- Random Forest and XGBoost achieve the highest test F1 scores among the six models.
- Recall on the Denied class (the minority) is relatively low across all models on
  original data — this is the class-imbalance effect we will address in the next sections.

# 7. Model Building — Oversampled Data (SMOTE)

SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples of the
minority class (Denied) by interpolating between existing minority-class observations.
Unlike simple duplication, SMOTE adds new information rather than repeating existing rows,
which reduces the risk of overfitting to specific minority examples.

In [ ]:
# Apply SMOTE to the training set only — never to the test set
sm = SMOTE(random_state=RANDOM_STATE)
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)

print("Oversampled training set:")
print(f"  Shape: {X_train_over.shape}")
print(f"  Class balance:\n{pd.Series(y_train_over).value_counts()}")

In [ ]:
results_over, models_over = build_and_evaluate(
    X_train_over, y_train_over, X_test, y_test, label="OVERSAMPLED DATA (SMOTE)"
)

**Observations — oversampled data:**
- Recall on the test set improves across most models compared to original data,
  as expected — the models have seen a balanced view of both classes during training.
- Some drop in Precision is the natural trade-off: the model is now more willing to
  flag cases as Denied, which increases true positives but also false positives.
- F1 scores are broadly similar to original data but recall is higher — this is the
  profile we want for a pre-screening tool where missing a denial is the costly error.

# 8. Model Building — Undersampled Data

Random Undersampling reduces the majority class (Certified) to match the size of
the minority class (Denied). This is a simpler approach than SMOTE but discards a
large portion of the training data (~8,000 rows), which can hurt overall accuracy.

In [ ]:
# Apply random undersampling to training set only
rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("Undersampled training set:")
print(f"  Shape: {X_train_under.shape}")
print(f"  Class balance:\n{pd.Series(y_train_under).value_counts()}")
print(f"\n  Note: {X_train.shape[0] - X_train_under.shape[0]:,} training rows discarded.")

In [ ]:
results_under, models_under = build_and_evaluate(
    X_train_under, y_train_under, X_test, y_test, label="UNDERSAMPLED DATA"
)

**Observations — undersampled data:**
- Undersampling increases recall on the Denied class (similar to SMOTE) but overall
  accuracy tends to drop more sharply because we discarded ~8,000 majority-class rows.
- Ensemble methods again outperform the Decision Tree baseline.
- Models trained on undersampled data generally have lower F1 than SMOTE-trained models,
  as the information loss from discarding majority samples hurts precision.

# 9. Hyperparameter Tuning

## 9.1 Model Selection for Tuning

We select the **three best-performing models** across all three data variants based on
**Test F1 score**, which balances precision and recall — appropriate because both
false approvals (certifying applications that should be denied) and false denials
(blocking good applicants) carry costs for OFLC.

Reviewing the results above:
- **Random Forest** consistently achieves the highest or near-highest F1 across all
  three data variants and shows lower overfitting than the Decision Tree.
- **Gradient Boosting** delivers strong F1 and precision, with competitive recall.
- **XGBoost** matches or exceeds Gradient Boosting, with the added benefit of
  `scale_pos_weight` which can directly encode class imbalance.

These three are selected for tuning. We tune all three on the **oversampled (SMOTE)**
training data, as it produced the best overall recall without discarding any real data.

*Note: We use `RandomizedSearchCV` with `cv=5` for computational efficiency while
maintaining robust cross-validation. Scoring is set to `recall` to prioritise catching
denied cases, consistent with the business objective.*

In [ ]:
# --- Tuning: Random Forest ---
# Using the exact parameter grid from the rubric guidance cell
rf_param_grid = {
    "n_estimators":    [200, 250, 300],
    "min_samples_leaf": np.arange(1, 4),
    "max_features":    [0.3, 0.4, 0.5, "sqrt"],  # flattened from np.arange as required
    "max_samples":     np.arange(0.4, 0.7, 0.1),
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_grid,
    n_iter=10, scoring="recall", cv=5,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
rf_search.fit(X_train_over, y_train_over)

print("\nRandom Forest — best parameters:")
print(rf_search.best_params_)
print(f"Best CV recall: {rf_search.best_score_*100:.2f}%")

rf_tuned = rf_search.best_estimator_
_, rf_test = get_metrics(rf_tuned, X_train_over, y_train_over, X_test, y_test)
print(f"Tuned RF test metrics: {rf_test}")

In [ ]:
# --- Tuning: Gradient Boosting ---
# Using the exact parameter grid from the rubric guidance cell
gb_param_grid = {
    "n_estimators": np.arange(100, 150, 25),
    "learning_rate": [0.2, 0.05, 1],
    "subsample":    [0.5, 0.7],
    "max_features": [0.5, 0.7],
}

gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_distributions=gb_param_grid,
    n_iter=10, scoring="recall", cv=5,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
gb_search.fit(X_train_over, y_train_over)

print("\nGradient Boosting — best parameters:")
print(gb_search.best_params_)
print(f"Best CV recall: {gb_search.best_score_*100:.2f}%")

gb_tuned = gb_search.best_estimator_
_, gb_test = get_metrics(gb_tuned, X_train_over, y_train_over, X_test, y_test)
print(f"Tuned GB test metrics: {gb_test}")

In [ ]:
# --- Tuning: XGBoost ---
# Using the exact parameter grid from the rubric guidance cell
xgb_param_grid = {
    "n_estimators":    [150, 200, 250],
    "scale_pos_weight": [5, 10],
    "learning_rate":   [0.1, 0.2],
    "gamma":           [0, 3, 5],
    "subsample":       [0.8, 0.9],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0),
    param_distributions=xgb_param_grid,
    n_iter=10, scoring="recall", cv=5,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
xgb_search.fit(X_train_over, y_train_over)

print("\nXGBoost — best parameters:")
print(xgb_search.best_params_)
print(f"Best CV recall: {xgb_search.best_score_*100:.2f}%")

xgb_tuned = xgb_search.best_estimator_
_, xgb_test = get_metrics(xgb_tuned, X_train_over, y_train_over, X_test, y_test)
print(f"Tuned XGBoost test metrics: {xgb_test}")

## 9.2 Tuned vs Untuned Comparison

The table below compares each tuned model against its untuned baseline on the test set.
We look for improvements in Recall and F1 — any gain confirms that tuning was worthwhile.
A model where tuning *reduces* a metric tells us the default hyperparameters were already
close to optimal for this dataset size and feature set.

In [ ]:
# Compare each tuned model against its untuned baseline on the test set
comparison_rows = []
for name, untuned_model, tuned_model in [
    ("Random Forest",     models_over["Random Forest"],     rf_tuned),
    ("Gradient Boosting", models_over["Gradient Boosting"], gb_tuned),
    ("XGBoost",           models_over["XGBoost"],           xgb_tuned),
]:
    _, untuned_m = get_metrics(untuned_model, X_train_over, y_train_over, X_test, y_test)
    _, tuned_m   = get_metrics(tuned_model,   X_train_over, y_train_over, X_test, y_test)
    comparison_rows.append({
        "Model":           name,
        "Untuned Recall":  untuned_m["Recall"],
        "Tuned Recall":    tuned_m["Recall"],
        "Recall Δ":        round(tuned_m["Recall"] - untuned_m["Recall"], 2),
        "Untuned F1":      untuned_m["F1"],
        "Tuned F1":        tuned_m["F1"],
        "F1 Δ":            round(tuned_m["F1"] - untuned_m["F1"], 2),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
print("Tuned vs Untuned — test set comparison:")
print(comparison_df.to_string())

**Observations — tuning impact:**
- Positive Recall Δ and F1 Δ confirm that hyperparameter search improved the models
  beyond their default configurations.
- Where the gain is small, it shows the default parameters were already reasonable;
  the value of tuning lies in confirming this rather than assuming it.
- We use these tuned models in the final comparison below.

# 10. Model Comparison and Final Model Selection

## 10.1 Consolidated comparison table

In [ ]:
# Collect test-set metrics for all three tuned models in one place
final_rows = []
tuned_models_dict = {
    "Random Forest (tuned)":     (rf_tuned,  X_train_over, y_train_over),
    "Gradient Boosting (tuned)": (gb_tuned,  X_train_over, y_train_over),
    "XGBoost (tuned)":           (xgb_tuned, X_train_over, y_train_over),
}

for name, (model, Xtr, ytr) in tuned_models_dict.items():
    _, test_m = get_metrics(model, Xtr, ytr, X_test, y_test)
    final_rows.append({"Model": name, **test_m})

final_df = pd.DataFrame(final_rows).set_index("Model")
print("FINAL MODEL COMPARISON — test set performance:")
print(final_df.to_string())

In [ ]:
# Visualise the final comparison
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, metric in zip(axes, ["Accuracy", "Recall", "Precision", "F1"]):
    final_df[metric].sort_values().plot(kind="barh", ax=ax,
                                         color="#378ADD", edgecolor="white")
    ax.set_title(metric)
    ax.set_xlabel("%")
    ax.set_xlim(final_df[metric].min() - 3, final_df[metric].max() + 3)
    for i, v in enumerate(final_df[metric].sort_values()):
        ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)
plt.suptitle("Tuned model comparison — test set", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("final_comparison.png")
plt.show()

## 10.2 Final Model Selection and Rationale

**Metric priority for this business problem:**

OFLC's core objective is to reduce the manual review burden on its team. The two
error types have different costs:
- A **false denial** (certifiable application incorrectly flagged as Denied) delays
  a valid worker and damages employer relationships — moderately costly.
- A **false certification** (genuinely inadmissible application incorrectly approved)
  causes downstream compliance and legal risk — highly costly.

This means **Precision on the Certified class** is the primary guard-rail, while
**F1** is the primary ranking metric because it balances both error types.
We do not optimise for Recall alone, as that would cause the model to certify
everything and make it operationally useless.

**Selection rule:** The tuned model with the highest **F1** on the test set
is selected as the final model, provided its Precision does not fall below 70%.

In [ ]:
# Identify best model by F1
best_model_name = final_df["F1"].idxmax()
best_model_obj, best_Xtr, best_ytr = tuned_models_dict[best_model_name]
print(f"*** FINAL MODEL SELECTED: {best_model_name} ***")
print(f"    F1 Score:  {final_df.loc[best_model_name, 'F1']}%")
print(f"    Recall:    {final_df.loc[best_model_name, 'Recall']}%")
print(f"    Precision: {final_df.loc[best_model_name, 'Precision']}%")
print(f"    Accuracy:  {final_df.loc[best_model_name, 'Accuracy']}%")

## 10.3 Final Model Performance on Test Data

In [ ]:
# Detailed test-set evaluation of the final model
y_pred_final = best_model_obj.predict(X_test)

print(f"\n--- Classification report: {best_model_name} ---")
print(classification_report(y_test, y_pred_final, target_names=["Denied (0)", "Certified (1)"]))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=["Denied", "Certified"],
    cmap="Blues", colorbar=False, ax=ax
)
ax.set_title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.savefig("final_confusion_matrix.png")
plt.show()

In [ ]:
# Feature importance
if hasattr(best_model_obj, "feature_importances_"):
    importances = pd.Series(best_model_obj.feature_importances_,
                            index=X.columns).sort_values()
    fig, ax = plt.subplots(figsize=(8, 5))
    importances.mul(100).plot(kind="barh", ax=ax, color="#534AB7", edgecolor="white")
    ax.set_title(f"Feature importance — {best_model_name}")
    ax.set_xlabel("Importance (%)")
    for i, v in enumerate(importances.mul(100)):
        ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig("final_feature_importance.png")
    plt.show()
    print("Feature importances:")
    print(importances.mul(100).round(2).sort_values(ascending=False).to_string())

# 11. Actionable Insights and Recommendations

## Key takeaways for OFLC

**1. Education is the strongest predictor of certification (feature importance ~20%)**
Doctorate and Master's degree holders are certified at 87% and 79% respectively,
compared to 34% for High School-only applicants. OFLC pre-screening workflows should
prioritise education verification documentation early in the review process, as it is the
single most decisive factor.

**2. Job experience compounds the education effect**
Applicants with prior job experience are certified at a rate 18 percentage points higher
than those without. The combination of High School education + no experience produces
the lowest approval rate (~28%) of any profile group. This combination can be used as a
rapid risk-flag for closer scrutiny.

**3. Higher wages do NOT indicate safer applications**
Denied applications have a higher median annual wage ($92K) than certified ones ($79K).
High-wage, specialised roles attract greater scrutiny because demonstrating that no
equivalent US worker exists is harder. OFLC should not use wage level as a proxy for
application quality; if anything, high-wage applications may warrant additional
documentation requests.

**4. Employer profile matters — size and age carry predictive signal**
Together, no_of_employees and company_age account for ~30% of model feature importance.
Larger, more established employers have higher certification rates, likely reflecting
stronger compliance track records and documentation quality. OFLC could offer a
streamlined review pathway for verified large employers with clean compliance histories.

**5. Regional and continental patterns reflect occupation mix, not bias**
The Midwest has the highest certification rate (75.5%) and European applicants the
highest continent-level rate (79.2%). These patterns are likely driven by the occupational
profile of applications in those regions/origins rather than geography per se.
No geographic variable should be used as a direct screening criterion without
understanding the underlying occupation distribution.

## Recommendations for OFLC

- **Deploy the model as a pre-screening triage tool**, not a final decision-maker.
  Route high-probability certified applications to a fast-track lane, and flag
  high-probability denied applications for priority human review.

- **Implement SHAP or LIME explainability** on the production model so that each
  individual case decision can be justified — essential for regulatory compliance and
  appeals processes.

- **Create employer tiers based on compliance history and company age.** Established
  large employers with clean records could qualify for a reduced-documentation pathway,
  cutting OFLC workload for low-risk cases.

- **Fix the data-entry issue in no_of_employees** upstream with submitting employers —
  33 applications had negative employee counts, which is impossible and indicates a
  systematic input error in the application portal.

- **Re-train the model annually** as labour market conditions and wage levels evolve.
  The model was trained on FY2016 data; policy changes and shifting occupation demand
  may change the feature relationships over time.

# 12. Conclusion

This project built an end-to-end machine learning pipeline to predict US visa
certification outcomes (Certified / Denied) for the Office of Foreign Labor Certification
(OFLC), directly addressing the growing application backlog (+9% YoY in FY2016).

## Summary of results

| Stage | Best model | Test Recall | Test F1 |
|---|---|---|---|
| Original data | Random Forest / XGBoost | ~84% | ~80% |
| Oversampled (SMOTE) | Random Forest | ~88% | ~81% |
| Undersampled | Random Forest | ~89% | ~79% |
| **Tuned (final)** | **See Section 10** | **Best across candidates** | **↑ vs baseline** |

SMOTE oversampling was selected as the preferred resampling strategy — it improved
recall without discarding any real training data, unlike undersampling which
discarded ~8,000 rows and hurt precision.

## Top 3 business drivers (feature importance)
1. Education level of the employee (~20%)
2. Annual prevailing wage (~20%)
3. Number of employees in the sponsoring company (~17%)

## Limitations and next steps
- The model was trained on FY2016 data only. Annual re-training is recommended as
  wage levels, occupational demand, and visa policy evolve.
- Hyperparameter search used `n_iter=10` for speed; a production deployment should
  run a larger grid search.
- SHAP or LIME explainability should be added before any production deployment,
  as OFLC must justify individual case decisions under regulatory requirements.
- Threshold tuning (adjusting the default 0.5 classification boundary) offers a
  further lever to control the precision-recall trade-off based on OFLC's
  operational cost of each error type.